# Hybrid NID OCR: PaddleOCR (English) + PaddleOCR-VL (Bangla)

**Goal.** Read a Bangladeshi NID card by giving each script to the engine that handles it best:

| Text on the card | Engine | Why |
|---|---|---|
| English name, Date of Birth, NID number | PP-OCRv5 (`lang="en"`) | Small, fast on CPU, and it only picks characters it actually sees — no invented digits |
| Bangla name, father's name, mother's name, address | PaddleOCR-VL-1.6 | Classic PaddleOCR has **no** Bangla recognition model. The VL model added Bengali in version 1.5 |

**The approach (Option B — detect once, then route).**

```
card image
    |
    v
PP-OCR detection  ->  all text line boxes (script-agnostic)
    |
    v
PP-OCR English recognition on every line  ->  text + confidence
    |
    +--> line looks like clean English/digits  ->  KEEP this result
    |
    +--> line looks like misread Bangla       ->  crop it, send to VL  ->  Bangla text
    |
    v
merge -> map lines to fields -> regex / date / digit clean-up -> structured JSON
```

VL only ever sees a few small line crops, never the whole card. That matters a lot on a CPU-only laptop.

**Hardware reality (i5-10310U, 16 GB RAM, no NVIDIA GPU).** Everything runs on CPU. The Intel UHD graphics cannot be used by PaddlePaddle. PP-OCR is comfortable here; VL is a 0.9B model that generates text one token at a time, so it will be seconds per crop, not milliseconds. **These timings are for comparing engines on one machine — they are not production numbers.** Any real deployment of VL at 100–1000 TPS needs GPU servers, and that cost belongs in the final report.

**What the notebook measures** (same metrics as `benchmark.ipynb`, so results line up):
field-level exact match, character error rate (CER), latency p50 / p95 — plus two things specific to this notebook: how often routing sends a line to the wrong engine, and a **VL-only control run** to check whether the hybrid is actually earning its complexity.

---

## 0. One-time setup

Run these in **PowerShell**, not in the notebook.

### 0.1 Pick the venv

This notebook lives in `notebooks/`, next to `benchmark.ipynb` and `tesseract.ipynb`.

Because VL is reached over HTTP (see 0.2), **this notebook only needs classic PaddleOCR** — the heavy VL dependencies live in the separate `llama-server` process, not in the kernel. So `.venv-paddle` is enough:

```powershell
cd C:\path\to\OCR-RnD
.\.venv-paddle\Scripts\Activate.ps1

python -m pip install paddlepaddle==3.2.2      # CPU build, and PIN this version
python -m pip install -U paddleocr
python -m pip install requests pandas matplotlib opencv-python
```

Two things about that first line.

**CPU build.** Every install page shows `paddlepaddle-gpu==3.2.1` because they assume CUDA — that one is useless on your laptop.

**The pin is not optional.** PaddlePaddle **3.3.0 and 3.3.1 have a CPU regression** that crashes text detection with:

```
NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute
not support [pir::ArrayAttribute<pir::DoubleAttribute>]
(at ..\paddle\fluid\framework\new_executor\instruction\onednn\onednn_instruction.cc)
```

The bug is in the oneDNN path of the new PIR executor, so it hits any CPU inference, not just OCR. A fix is merged upstream but has not shipped to PyPI. **3.2.2 is the newest release without it.**

If you cannot downgrade right now, set `MKLDNN_WORKAROUND = True` in the config cell and restart the kernel. That turns oneDNN off and dodges the crash — but oneDNN is exactly what makes PP-OCR fast on CPU, so your latency numbers stop lining up with `benchmark.ipynb`. For a benchmarking notebook, downgrade rather than work around.

Use `.venv-paddlevl` instead **only** if you end up on the `transformers` fallback in section 2b, since that one loads the VL model into the kernel itself and needs `paddleocr[doc-parser]>=3.6.0`.

### 0.2 Install llama.cpp (this is the fast path for VL on CPU)

You asked for whichever backend gives faster output. On a CPU-only machine that is **llama.cpp with the GGUF model**, not the default Paddle/transformers backend. llama.cpp is built for CPU inference and the PaddleOCR team ships official GGUF weights.

```powershell
winget install llama.cpp
```

### 0.3 Download the model (about 1 GB, one time)

From <https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6-GGUF> download two files:

* the main model, e.g. `PaddleOCR-VL-1.6-Q4_K_M.gguf`
* the vision projector, the file with **`mmproj`** in its name

Both are needed. The mmproj file is the part that lets the model see images.

Put them **outside the repo** — `C:\models\` is fine. A 1 GB file inside `OCR-RnD` will bloat the git history, and your `.gitignore` does not cover it.

### 0.4 Start the server, and leave it running

```powershell
llama-server -m C:\models\PaddleOCR-VL-1.6-Q4_K_M.gguf --mmproj C:\models\PaddleOCR-VL-1.6-mmproj.gguf --port 8080 --host 127.0.0.1 --temp 0 -t 4
```

`-t 4` = 4 threads, one per physical core. `--temp 0` makes output repeatable, which you want for a benchmark.

Keep this window open the whole time you use the notebook.

> **If llama.cpp does not work out**, there is a `transformers` fallback in section 2b. It is slower, but it needs no extra install.

---

## 1. Imports and configuration

Everything you might need to change lives in this one cell.

In [ ]:
import base64
import io
import json
import os
import re
import time
from pathlib import Path

# ----------------------------------------------------------------------
# PaddlePaddle 3.3.x has a CPU regression that crashes text detection with
#   NotImplementedError: ConvertPirAttribute2RuntimeAttribute not support
#     [pir::ArrayAttribute<pir::DoubleAttribute>]  (onednn_instruction.cc)
#
# Proper fix is to pin the framework (see the markdown cell above):
#     pip install paddlepaddle==3.2.2
#
# If you cannot downgrade right now, set this to True. It turns oneDNN off,
# which avoids the crash but makes PP-OCR slower on CPU - so the latency
# numbers will NOT be comparable with benchmark.ipynb. Downgrade instead.
# These must be set before paddle is imported, hence their place here.
# ----------------------------------------------------------------------
MKLDNN_WORKAROUND = False

if MKLDNN_WORKAROUND:
    os.environ["FLAGS_use_mkldnn"] = "0"
    os.environ["FLAGS_enable_pir_in_executor"] = "0"

import cv2
import numpy as np
import pandas as pd
import requests
from PIL import Image

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

# ----------------------------------------------------------------------
# CONFIG - edit these
# ----------------------------------------------------------------------
# This notebook lives in OCR-RnD/notebooks/, so every data path is one
# level up. Rather than hard-coding "..", walk up until data/data.csv is
# found - that way the notebook also works if run from the repo root.
def find_project_root(start: Path = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "data.csv").exists():
            return p
    raise FileNotFoundError("could not find OCR-RnD root (no data/data.csv above cwd)")

PROJECT_DIR   = find_project_root()
DATA_DIR      = PROJECT_DIR / "data"

# Two image folders exist. images_1600/ is the same 20 cards capped at
# 1600px on the long side. Use it: the full-size set contains 3a.jpg at
# 4608x3456, and PP-OCR's detector defaults to limit_type="min", so it
# does not downscale - it runs DBNet at 4000x3000 and dies with
#   RuntimeError: Unknown exception        (allocation failure in predictor.run)
# Note bench/common.py still points at data/images, so switch benchmark.ipynb
# too if you want the numbers to stay comparable.
# IMAGE_DIR   = DATA_DIR / "images"
IMAGE_DIR     = DATA_DIR / "images_1600"

GT_CSV        = DATA_DIR / "data.csv"

# results/ exists both at the repo root and inside notebooks/.
# Root is the default here; switch if benchmark.ipynb writes to the other.
OUT_DIR       = PROJECT_DIR / "results"
# OUT_DIR     = Path("results")            # i.e. notebooks/results/
OUT_DIR.mkdir(parents=True, exist_ok=True)

VL_SERVER     = "http://127.0.0.1:8080/v1"      # llama-server from step 0.4

# Routing thresholds - section 4 explains these and section 4b tunes them
SCORE_THRESHOLD   = 0.80   # below this confidence, suspect the English model misread it
ASCII_RATIO_MIN   = 0.55   # below this share of plain ASCII, suspect non-Latin script
CROP_PAD          = 6      # pixels of padding around each line crop
CROP_MIN_HEIGHT   = 48     # upscale crops shorter than this before sending to VL
DET_MAX_SIDE      = 1600   # downscale any card longer than this before detection

N_BENCH_CARDS     = 20     # cards for the main benchmark
N_CONTROL_CARDS   = 5      # cards for the slow VL-only control run

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def _count(d: Path) -> str:
    if not d.exists():
        return "MISSING"
    return f"{sum(1 for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)} images"


print("project root  :", PROJECT_DIR)
print("images        :", IMAGE_DIR, "-", _count(IMAGE_DIR))
print("other set     :", DATA_DIR / "images_1600", "-", _count(DATA_DIR / "images_1600"))
print("ground truth  :", GT_CSV, "-", "found" if GT_CSV.exists() else "MISSING")
print("results out   :", OUT_DIR)

---

## 2. Load the English engine

PP-OCRv5 with `lang="en"`. Three helper modules are turned off on purpose:

* `use_doc_orientation_classify` — decides if the page is upside down. Extra model, extra time, and NID photos are rarely rotated 180°.
* `use_doc_unwarping` — flattens curled paper. Another extra model; we will handle skew with a perspective crop instead.
* `use_textline_orientation` — checks if individual lines are rotated 90°. Rare on ID cards.

Leaving them off keeps CPU time down. Turn them back on later if you see failures that they would fix.

The first run downloads model files (a few tens of MB) and is slow. Later runs are fast.

In [ ]:
import paddle
import paddleocr
from paddleocr import PaddleOCR

# --- version check: catch the 3.3.x CPU regression before it crashes ----
pp_ver = tuple(int(x) for x in paddle.__version__.split(".")[:2])
print(f"paddlepaddle {paddle.__version__} | paddleocr {paddleocr.__version__}")

if pp_ver >= (3, 3) and not MKLDNN_WORKAROUND:
    print("\n" + "!" * 70)
    print("paddlepaddle 3.3.x has a CPU bug that will crash text detection.")
    print("Fix properly:   pip install paddlepaddle==3.2.2")
    print("Or, to limp on: set MKLDNN_WORKAROUND = True in the config cell,")
    print("                then restart the kernel. Slower, so latency numbers")
    print("                stop being comparable with benchmark.ipynb.")
    print("!" * 70 + "\n")

kwargs = dict(
    lang="en",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
if MKLDNN_WORKAROUND:
    kwargs["enable_mkldnn"] = False

t0 = time.perf_counter()
ocr_en = PaddleOCR(**kwargs)
print(f"PP-OCRv5 (en) ready in {time.perf_counter() - t0:.1f}s"
      f"{'  [oneDNN OFF - slower]' if MKLDNN_WORKAROUND else ''}")

### 2a. Connect to the VL model (llama.cpp — the fast path)

`llama-server` exposes an OpenAI-style API, so we just POST the crop as a base64 image plus a prompt.

The prompt matters. PaddleOCR-VL is trained with specific task prompts, and `"OCR:"` is the one for plain text recognition of a single element. Do not replace it with a chatty instruction like "please read this text" — the model was not trained that way and quality drops.

In [ ]:
VL_PROMPTS = {
    "ocr":      "OCR:",       # plain text in a cropped element  <- what we use
    "spotting": "Spotting:",  # find AND read all text in a larger image
    "table":    "Table Recognition:",
    "formula":  "Formula Recognition:",
}


def pil_to_data_url(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def vl_ocr_llamacpp(img: Image.Image, task: str = "ocr",
                    server: str = VL_SERVER, max_tokens: int = 256) -> str:
    payload = {
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": pil_to_data_url(img)}},
                {"type": "text", "text": VL_PROMPTS[task]},
            ],
        }],
        "temperature": 0.0,
        "max_tokens": max_tokens,
        # Greedy decoding with no repetition penalty falls into a token loop on
        # whole-card Spotting: 1024 tokens of "gaqaqaqa...", 53s, one usable
        # line. 1.1 stops it without affecting short line crops.
        "repeat_penalty": 1.1,
    }
    r = requests.post(f"{server}/chat/completions", json=payload, timeout=600)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()


# --- smoke test: a white strip with Bangla text drawn on it -------------
def _smoke_test():
    try:
        r = requests.get(f"{VL_SERVER}/models", timeout=10)
        r.raise_for_status()
        print("server is up:", [m.get("id") for m in r.json().get("data", [])])
    except Exception as e:
        print("CANNOT REACH THE VL SERVER.")
        print("Is `llama-server` still running in PowerShell? See setup step 0.4.")
        print("error:", e)
        return
    test = Image.new("RGB", (420, 80), "white")
    t0 = time.perf_counter()
    out = vl_ocr_llamacpp(test)
    print(f"blank-image round trip: {time.perf_counter() - t0:.2f}s -> {out!r}")
    print("(empty or near-empty output on a blank image is the correct result)")

_smoke_test()

### 2b. Fallback: VL through `transformers`

Only run this cell if llama.cpp did not work. It loads the model into RAM in this process (about 2 GB in float32 on CPU) and will be noticeably slower per crop.

In [ ]:
# LEAVE THIS CELL UNRUN unless llama.cpp failed.
USE_TRANSFORMERS_FALLBACK = False

if USE_TRANSFORMERS_FALLBACK:
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText

    _MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.6"
    _model = AutoModelForImageTextToText.from_pretrained(
        _MODEL_ID, torch_dtype=torch.float32   # CPU has no usable bfloat16 here
    ).eval()
    _proc = AutoProcessor.from_pretrained(_MODEL_ID)

    def vl_ocr_transformers(img: Image.Image, task: str = "ocr", max_tokens: int = 256) -> str:
        messages = [{"role": "user", "content": [
            {"type": "image", "image": img.convert("RGB")},
            {"type": "text",  "text": VL_PROMPTS[task]},
        ]}]
        inputs = _proc.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
            images_kwargs={"size": {"shortest_edge": _proc.image_processor.min_pixels,
                                    "longest_edge": 1280 * 28 * 28}},
        )
        out = _model.generate(**inputs, max_new_tokens=max_tokens)
        return _proc.decode(out[0][inputs["input_ids"].shape[-1]:-1]).strip()

    print("transformers fallback loaded")


# One name for the rest of the notebook, whichever backend is active.
def vl_ocr(img: Image.Image, task: str = "ocr", max_tokens: int = 256) -> str:
    if USE_TRANSFORMERS_FALLBACK:
        return vl_ocr_transformers(img, task, max_tokens)
    return vl_ocr_llamacpp(img, task, max_tokens=max_tokens)

print("active VL backend:", "transformers" if USE_TRANSFORMERS_FALLBACK else "llama.cpp")

---

## 3. Step 1 — detect and read every line with PP-OCR

Detection in PP-OCR finds **where** text is. It works on pixel shapes, not on language, so it finds Bangla lines just as well as English ones. That is exactly why Option B works: one detector, then we choose the recognizer per line.

Recognition is the part that is English-only. It will produce nonsense on Bangla lines — and that nonsense is our routing signal.

In [ ]:
def list_cards(n=None):
    files = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)
    return files[:n] if n else files


def run_ppocr(image_path):
    """Return list of dicts: {poly, text, score} plus wall-clock seconds."""
    t0 = time.perf_counter()
    # limit_type="max" makes the detector downscale anything longer than
    # limit_side_len. Without it PP-OCR leaves a high-res photo at native
    # size and the detector blows up on the allocation - see cell 3.
    result = ocr_en.predict(str(image_path),
                            text_det_limit_type="max",
                            text_det_limit_side_len=DET_MAX_SIDE)
    elapsed = time.perf_counter() - t0

    lines = []
    for res in result:
        d = res if isinstance(res, dict) else getattr(res, "json", {}).get("res", res)
        polys  = d.get("dt_polys", [])
        texts  = d.get("rec_texts", [])
        scores = d.get("rec_scores", [])
        for poly, text, score in zip(polys, texts, scores):
            lines.append({
                "poly":  np.asarray(poly, dtype=np.float32),
                "text":  text,
                "score": float(score),
            })
    return lines, elapsed


cards = list_cards()
print(f"{len(cards)} card images found")
for i, c in enumerate(cards):
    print(i, c.name)

sample = cards[12]
lines, t_det = run_ppocr(sample)
print(f"\n{sample.name}: {len(lines)} lines in {t_det:.2f}s\n")
for i, ln in enumerate(lines):
    print(f"{i:2d}  {ln['score']:.2f}  {ln['text']!r}")

Look at that output before going on. You should see two clusters: clean readable English lines with high confidence, and garbage strings where a Bangla line was forced through the English model. That gap is what section 4 turns into a rule.

In [ ]:
def draw_lines(image_path, lines, labels=None, title=""):
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    vis = img.copy()
    for i, ln in enumerate(lines):
        colour = (0, 160, 0)                       # green = English by default
        if labels is not None and labels[i] == "bangla":
            colour = (220, 0, 0)                   # red = routed to VL
        cv2.polylines(vis, [ln["poly"].astype(np.int32)], True, colour, 2)
        x, y = ln["poly"][0]
        cv2.putText(vis, str(i), (int(x), max(int(y) - 4, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1, cv2.LINE_AA)
    plt.figure(figsize=(11, 8))
    plt.imshow(vis); plt.axis("off"); plt.title(title or image_path.name)
    plt.show()

draw_lines(sample, lines, title="PP-OCR detected lines")

---

## 4. Step 2 — the routing rule

For each line, decide: keep the English result, or crop it and send it to VL?

Three signals:

1. **Confidence.** The English recognizer reports how sure it is. On Bangla shapes it is usually unsure.
2. **ASCII ratio.** Real English output is nearly all plain letters, digits and spaces. Misread Bangla tends to come back with odd symbols and mixed case junk.
3. **A protective override.** If a line is a clean run of digits, slashes and dashes — an NID number or a date — we keep it as English no matter what the other two signals say. These are the two fields where a mistake is a compliance problem, and the non-generative engine is the safer reader.

A line goes to VL when signal 1 **or** signal 2 fires, unless the override applies.

In [ ]:
BANGLA_RANGE = re.compile(r"[\u0980-\u09FF]")          # Bangla Unicode block
DIGIT_LINE   = re.compile(r"^[\s0-9/\-.:]+$")          # NID number, or 15/08/1998
DATE_LINE    = re.compile(                             # or 01 Jan 1990
    r"^[\s0-9/\-.:]*\d{1,2}[\s/\-.]*[A-Za-z]{3,9}[\s/\-.]*\d{2,4}[\s0-9/\-.:]*$")


def ascii_ratio(text: str) -> float:
    if not text:
        return 0.0
    ok = sum(1 for ch in text if ch.isascii() and (ch.isalnum() or ch in " .,:/-'()"))
    return ok / len(text)


def route_line(line) -> str:
    """Return 'english' (keep PP-OCR result) or 'bangla' (send crop to VL)."""
    text, score = line["text"], line["score"]

    # protective override: never hand a number or a date to a generative model,
    # even when confidence is low - these are the KYC-critical fields
    t = (text or "").strip()
    if len(t) >= 4 and score >= 0.40 and (DIGIT_LINE.match(t) or DATE_LINE.match(t)):
        return "english"

    if score < SCORE_THRESHOLD:
        return "bangla"
    if ascii_ratio(text) < ASCII_RATIO_MIN:
        return "bangla"
    return "english"


labels = [route_line(ln) for ln in lines]

print(f"{'#':>2}  {'route':<8} {'conf':>5} {'ascii':>6}  text")
print("-" * 70)
for i, (ln, lab) in enumerate(zip(lines, labels)):
    print(f"{i:2d}  {lab:<8} {ln['score']:>5.2f} {ascii_ratio(ln['text']):>6.2f}  {ln['text']!r}")

print(f"\n-> {labels.count('bangla')} of {len(labels)} lines go to VL")
draw_lines(sample, lines, labels, title="green = kept as English   |   red = sent to VL")

### 4b. Sanity-check the thresholds

Routing errors are the main risk in this design, so measure them rather than trusting the defaults. Look at the picture above and count by eye on 2–3 cards:

* a **Bangla line kept as English** — the worst case, you get garbage in a name field
* an **English line sent to VL** — wasteful and slow, and VL might paraphrase rather than transcribe

If Bangla lines are leaking through, raise `SCORE_THRESHOLD`. If clean English lines are being sent to VL, lower it. The cell below shows how the split would change at different thresholds so you can pick a value instead of guessing.

In [ ]:
probe_cards = cards[:3]
rows = []
for path in probe_cards:
    ls, _ = run_ppocr(path)
    for thr in [0.70, 0.75, 0.80, 0.85, 0.90]:
        old = SCORE_THRESHOLD
        globals()["SCORE_THRESHOLD"] = thr
        n_vl = sum(1 for l in ls if route_line(l) == "bangla")
        globals()["SCORE_THRESHOLD"] = old
        rows.append({"card": path.name, "threshold": thr,
                     "lines": len(ls), "sent_to_vl": n_vl})

pivot = pd.DataFrame(rows).pivot(index="threshold", columns="card", values="sent_to_vl")
print("lines sent to VL, by confidence threshold\n")
print(pivot)
print("\nPick the threshold where the count matches the number of Bangla lines you can see on the card.")

---

## 5. Step 3 — crop the Bangla lines and read them with VL

Two details make a real difference to VL accuracy on crops:

* **Perspective crop, not a rectangle.** Phone photos of ID cards are skewed. PP-OCR returns a four-point polygon that follows the tilt of the line. Warping that polygon to a straight rectangle hands VL an upright line instead of a slanted one inside a box full of neighbouring text.
* **Upscale short crops.** A line 20 pixels tall carries little detail. Enlarging it to about 48 pixels tall costs almost nothing and gives the vision encoder more to work with.

In [ ]:
def crop_line(image_bgr, poly, pad=CROP_PAD, min_h=CROP_MIN_HEIGHT) -> Image.Image:
    """Perspective-warp a 4-point polygon into an upright crop."""
    pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)

    # order corners: top-left, top-right, bottom-right, bottom-left
    s, d = pts.sum(axis=1), np.diff(pts, axis=1).ravel()
    box = np.array([pts[np.argmin(s)], pts[np.argmin(d)],
                    pts[np.argmax(s)], pts[np.argmax(d)]], dtype=np.float32)

    w = int(max(np.linalg.norm(box[0] - box[1]), np.linalg.norm(box[3] - box[2]))) + 2 * pad
    h = int(max(np.linalg.norm(box[0] - box[3]), np.linalg.norm(box[1] - box[2]))) + 2 * pad
    w, h = max(w, 8), max(h, 8)

    dst = np.array([[pad, pad], [w - pad, pad], [w - pad, h - pad], [pad, h - pad]],
                   dtype=np.float32)
    M = cv2.getPerspectiveTransform(box, dst)
    warped = cv2.warpPerspective(image_bgr, M, (w, h), borderValue=(255, 255, 255))

    if h < min_h:                       # upscale small crops
        scale = min_h / h
        warped = cv2.resize(warped, (int(w * scale), min_h), interpolation=cv2.INTER_CUBIC)

    return Image.fromarray(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))


# run VL on every line the router flagged on the sample card
image_bgr = cv2.imread(str(sample))
vl_results, vl_times = {}, []

for i, (ln, lab) in enumerate(zip(lines, labels)):
    if lab != "bangla":
        continue
    crop = crop_line(image_bgr, ln["poly"])
    t0 = time.perf_counter()
    text = vl_ocr(crop)
    dt = time.perf_counter() - t0
    vl_times.append(dt)
    vl_results[i] = text

    plt.figure(figsize=(7, 1.4)); plt.imshow(crop); plt.axis("off"); plt.show()
    print(f"line {i}   PP-OCR(en): {ln['text']!r}")
    print(f"          VL Bangla : {text!r}   ({dt:.2f}s)\n")

if vl_times:
    print(f"VL: {len(vl_times)} crops, {np.mean(vl_times):.2f}s average, "
          f"{sum(vl_times):.1f}s total on this card")

**Read those pairs carefully.** This is the single most important cell in the notebook. If the VL column shows correct Bangla where PP-OCR showed garbage, the hybrid idea works and everything after this is measurement. If VL also struggles, stop here and tell your supervisor — no amount of pipeline work fixes a recognizer that cannot read the script.

Also watch the per-crop time. Multiply it by the number of Bangla lines to see what one card costs, and that is your ceiling on this hardware.

---

## 6. Step 4 — turn lines into fields

Raw text lines are not the deliverable. The MFS sign-up flow needs structured fields.

Strategy: find a **label** (`Name`, `Date of Birth`, `ID NO`, `নাম`, `পিতা`, `মাতা`), then take the value either from the rest of that same line or from the line sitting immediately to its right / below it. Labels are matched loosely because OCR mangles them — `Dale of Birth` and `1D NO` are common.

Post-processing then does three jobs:

* **Bangla digits to ASCII.** `০১২৩৪৫৬৭৮৯` → `0123456789`. Cards mix both.
* **NID number check.** Must be 10, 13 or 17 digits once spaces are stripped. A wrong length is *reported* in a `_flags` list, never quietly returned as if it were fine — in a KYC pipeline a field that failed its own check has to be visible.
* **Date parsing.** Cards print dates like `01 Jan 1990` or `01/01/1990`. Normalise to `YYYY-MM-DD`, and repair the classic OCR digit confusions (`O`→`0`, `l`/`I`→`1`, `S`→`5`, `B`→`8`) **only inside** date and number fields, never inside names.

In [ ]:
BN2EN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
DIGIT_FIX    = str.maketrans({"O": "0", "o": "0", "l": "1", "I": "1", "|": "1",
                              "S": "5", "B": "8", "Z": "2", "b": "6"})

# Valid NID lengths. 12 is deliberately absent: the two 12-digit values in
# data.csv are missing a leading zero (Excel dropped it); the cards print 13.
ACCEPTED_NID_LENGTHS = (10, 13, 15, 17)

# Label patterns per field. "National ID" is NOT here any more - it matched
# the card's title "National ID Card" and stole the NID slot on card 110.
LABELS = {
    "name_en":   [r"\bn[a@]m[e3]\b"],
    "dob":       [r"date\s*of\s*b[il1]rth", r"\bd\.?o\.?b\b", r"da[lt]e\s*of"],
    "nid":       [r"\bn[il1]d\s*n[o0]", r"\b[il1]d\s*n[o0]\b"],
    "name_bn":   [r"নাম"],
    "father_bn": [r"পিতা"],
    "mother_bn": [r"মাতা"],
}

# Header text on every card - never a field value
HEADER_WORDS_BN = ["প্রজাতন্ত্র", "বাংলাদেশ", "সরকার", "পরিচয়", "জাতীয়"]
HEADER_WORDS_EN = re.compile(r"government|republic|bangladesh|national|\bcard\b", re.I)

# Scripts that can never appear on a Bangladeshi NID. If VL outputs these,
# it made them up (seen on card 110: Tibetan "ལྟར", Chinese "同").
FOREIGN_SCRIPT = re.compile(r"[\u0F00-\u0FFF\u4E00-\u9FFF\u3040-\u30FF\uAC00-\uD7AF]")

BANGLA_LETTER = re.compile(r"[\u0980-\u09FF]")

MONTHS = {m.lower(): i for i, m in enumerate(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}


# ---------------------------------------------------------------------------
# small cleaners
# ---------------------------------------------------------------------------
def norm_digits(s: str) -> str:
    return (s or "").translate(BN2EN_DIGITS)


def clean_nid(s: str):
    """Return (digits, is_valid). A wrong length is reported, never hidden."""
    digits = re.sub(r"\D", "", norm_digits(s).translate(DIGIT_FIX))
    if not digits:
        return None, False
    return digits, len(digits) in ACCEPTED_NID_LENGTHS


def clean_dob(s: str):
    s = norm_digits(s or "")
    # 01 May 1977 / 01-May-1977 / 01May1977
    m = re.search(r"(\d{1,2})\s*[/\-. ]?\s*([A-Za-z]{3,})\s*[/\-. ]?\s*(\d{4})", s)
    if m:
        mon = MONTHS.get(m.group(2)[:3].lower())
        if mon and 1 <= int(m.group(1)) <= 31:
            return f"{int(m.group(3)):04d}-{mon:02d}-{int(m.group(1)):02d}"
    # 01/05/1977
    m = re.search(r"(\d{1,2})\s*[/\-.]\s*(\d{1,2})\s*[/\-.]\s*(\d{4})", s)
    if m and 1 <= int(m.group(1)) <= 31 and 1 <= int(m.group(2)) <= 12:
        return f"{int(m.group(3)):04d}-{int(m.group(2)):02d}-{int(m.group(1)):02d}"
    return None


def strip_label(text: str, patterns) -> str:
    out = text
    for p in patterns:
        out = re.sub(p + r"\s*[:：ঃ.\-]*", "", out, flags=re.I)
    return out.strip(" :：ঃ.-/")


def labels_in(text: str):
    """Which fields' labels appear in this text."""
    return [f for f, pats in LABELS.items()
            if any(re.search(p, text or "", flags=re.I) for p in pats)]


def is_header(text: str) -> bool:
    return any(w in text for w in HEADER_WORDS_BN) or bool(HEADER_WORDS_EN.search(text))


# ---------------------------------------------------------------------------
# "does this look like a real value for this field?"
# Used so the search never accepts junk like "2018" as a name.
# ---------------------------------------------------------------------------
def plausible(field: str, value: str) -> bool:
    v = (value or "").strip()
    if not v or labels_in(v) or is_header(v):
        return False
    if field == "name_en":
        letters = re.findall(r"[A-Za-z]", v)
        return (len(letters) >= 3 and not BANGLA_LETTER.search(v)
                and not re.search(r"\d", v))
    if field in ("name_bn", "father_bn", "mother_bn"):
        return len(BANGLA_LETTER.findall(v)) >= 2
    if field == "dob":
        return clean_dob(v) is not None
    if field == "nid":
        return len(re.sub(r"\D", "", norm_digits(v))) >= 8
    return False


# ---------------------------------------------------------------------------
# main function
# ---------------------------------------------------------------------------
def extract_fields(merged):
    """merged: list of {text, poly, source}. Returns a field dict.

    How it finds each field, in order of preference:
      1. label and value on the same line       "Date of Birth 01 May 1977"
      2. label alone, value to its RIGHT         "NID No" | "776 332 2729"
      3. label alone, value just BELOW it        "নাম" above "রাম প্রসাদ ঘোষ"
      4. no label at all:
           dob / nid  -> any line that parses as a date / NID number
           Bangla     -> by position, if the number of Bangla name lines
                         matches the number of missing Bangla fields
                         (they always come in the order name, father, mother)

    Distances are measured from box EDGES and scaled to the card's own line
    height, so it works on a 500 px photocopy and a 1600 px photo alike.
    """
    FIELDS_ORDER = ["nid", "dob", "name_en", "name_bn", "father_bn", "mother_bn"]
    fields = {k: None for k in ["name_en", "name_bn", "father_bn", "mother_bn", "dob", "nid"]}
    how, flags = {}, []

    # ---- 1. build clean boxes --------------------------------------------
    boxes = []
    for i, ln in enumerate(merged):
        text = (ln.get("text") or "").strip()
        if not text or FOREIGN_SCRIPT.search(text):
            continue                                  # empty or made-up script
        p = np.asarray(ln["poly"], dtype=np.float32).reshape(-1, 2)
        x0, y0 = p.min(axis=0)
        x1, y1 = p.max(axis=0)
        boxes.append({"i": i, "text": text,
                      "x0": x0, "y0": y0, "x1": x1, "y1": y1,
                      "h": max(y1 - y0, 1.0), "w": max(x1 - x0, 1.0)})
    if not boxes:
        for k in fields:
            flags.append(f"{k} not found")
        fields["_flags"], fields["_how"] = flags, how
        return fields

    h_med = float(np.median([b["h"] for b in boxes]))
    W = float(max(b["x1"] for b in boxes) - min(b["x0"] for b in boxes))
    used = set()                                       # box indices already taken

    def same_row(a, b):
        overlap = min(a["y1"], b["y1"]) - max(a["y0"], b["y0"])
        return overlap > 0.5 * min(a["h"], b["h"])

    def right_of(label, c):
        return same_row(label, c) and c["x0"] >= label["x1"] - 0.02 * W

    def below(label, c):
        gap = c["y0"] - label["y1"]
        if not (-0.3 * h_med <= gap < 1.5 * h_med):
            return None
        aligned = abs(c["x0"] - label["x0"]) < 0.25 * W
        h_overlap = min(label["x1"], c["x1"]) - max(label["x0"], c["x0"])
        if aligned or h_overlap > 0.3 * min(label["w"], c["w"]):
            return gap
        return None

    def take(field, box, value, method):
        fields[field] = value
        how[field] = method
        used.add(box["i"])

    # ---- 2. label-based search -------------------------------------------
    for field in FIELDS_ORDER:
        pats = LABELS[field]
        label_boxes = [b for b in boxes if field in labels_in(b["text"])]

        for lb in label_boxes:                          # try every label hit,
            if fields[field] is not None:               # not just the first
                break

            # (a) value on the same line as the label
            rest = strip_label(lb["text"], pats)
            if plausible(field, rest):
                take(field, lb, rest, "same line")
                break

            # (b) value to the right, (c) value below
            right, down = [], []
            for c in boxes:
                if c is lb or c["i"] in used or not plausible(field, c["text"]):
                    continue
                if right_of(lb, c):
                    right.append((c["x0"] - lb["x1"], c))
                else:
                    g = below(lb, c)
                    if g is not None:
                        down.append((g, c))
            if right:
                c = min(right, key=lambda t: t[0])[1]
                take(field, c, c["text"], "right of label")
            elif down:
                c = min(down, key=lambda t: t[0])[1]
                take(field, c, c["text"], "below label")

    # ---- 3a. no label found: dob / nid by content ---------------------------
    if fields["dob"] is None:
        for b in boxes:
            if b["i"] not in used and clean_dob(b["text"]):
                take("dob", b, b["text"], "no label - looks like a date")
                break

    if fields["nid"] is None:
        for b in boxes:
            if b["i"] in used or clean_dob(b["text"]):
                continue
            real_digits = re.sub(r"\D", "", norm_digits(b["text"]))
            if len(real_digits) in ACCEPTED_NID_LENGTHS:
                take("nid", b, b["text"], "no label - looks like an NID")
                break

    # ---- 3b. no label found: Bangla names by position -----------------------
    # Every NID lists them top to bottom as name, father, mother. Only used
    # when the number of candidate lines matches the number of missing
    # fields exactly, so it cannot guess wildly.
    bn_fields = ["name_bn", "father_bn", "mother_bn"]
    if any(fields[f] is None for f in bn_fields):
        stop_y = min([b["y0"] for b in boxes
                      if clean_dob(b["text"]) or "dob" in labels_in(b["text"])
                      or "nid" in labels_in(b["text"])] or [float("inf")])
        rows = []
        for b in boxes:
            if b["y0"] >= stop_y or is_header(b["text"]) or b["i"] in used:
                continue
            label_here = [f for f in labels_in(b["text"]) if f in bn_fields]
            val = strip_label(b["text"], LABELS[label_here[0]]) if label_here else b["text"]
            if plausible("name_bn", val):
                rows.append((b["y0"], b, val))

        # Drop label-sized lines. On smart cards the labels নাম / পিতা / মাতা
        # are one short word on their own line, and VL often misreads them
        # into a different short Bangla word (card 21: মাতা -> "রাজ"). A real
        # name line is far wider, so anything under 40% of the typical width
        # is treated as a label, not a name.
        if rows:
            typical_w = float(np.median([b["w"] for _, b, _ in rows]))
            rows = [r for r in rows if r[1]["w"] >= 0.4 * typical_w]
        rows.sort(key=lambda t: t[0])

        missing = [f for f in bn_fields if fields[f] is None]
        if len(rows) == len(missing) and missing:
            for f, (_, b, val) in zip(missing, rows):
                take(f, b, val, "by position (label not read)")

    # ---- 4. final clean-up and flags ---------------------------------------
    if fields["nid"]:
        fields["nid"], ok = clean_nid(fields["nid"])
        if not ok:
            flags.append(f"nid has {len(fields['nid'] or '')} digits, "
                         f"expected {'/'.join(map(str, ACCEPTED_NID_LENGTHS))}")
    if fields["dob"]:
        raw_dob = fields["dob"]
        fields["dob"] = clean_dob(raw_dob)
        if fields["dob"] is None:
            flags.append(f"dob not parseable from {raw_dob!r}")

    for k in ["name_en", "name_bn", "father_bn", "mother_bn", "dob", "nid"]:
        if fields[k] is None:
            flags.append(f"{k} not found")
        elif not how.get(k, "").startswith(("same line", "right", "below")):
            flags.append(f"{k}: {how[k]}")

    fields["_flags"] = flags
    fields["_how"] = how          # which rule found each field - for debugging
    return fields

---

## 7. Step 5 — the whole pipeline in one function

Everything above, wired together, with a timing breakdown per stage. The timing split is what tells you where the seconds actually go.

In [ ]:
def hybrid_ocr(image_path, verbose=False):
    timings = {}
    t_start = time.perf_counter()

    lines, t_ppocr = run_ppocr(image_path)
    timings["ppocr_detect_recognise"] = t_ppocr

    labs = [route_line(l) for l in lines]
    image_bgr = cv2.imread(str(image_path))

    merged, t_vl = [], 0.0
    for ln, lab in zip(lines, labs):
        if lab == "bangla":
            crop = crop_line(image_bgr, ln["poly"])
            t0 = time.perf_counter()
            try:
                text = vl_ocr(crop)
            except Exception as e:
                text, lab = ln["text"], "english(vl-failed)"
                if verbose:
                    print("VL call failed:", e)
            t_vl += time.perf_counter() - t0
            merged.append({"text": text, "poly": ln["poly"], "source": "vl",
                           "ppocr_text": ln["text"], "score": ln["score"]})
        else:
            merged.append({"text": ln["text"], "poly": ln["poly"], "source": "ppocr",
                           "ppocr_text": ln["text"], "score": ln["score"]})

    timings["vl_bangla_crops"] = t_vl
    timings["n_lines"] = len(lines)
    timings["n_vl_crops"] = labs.count("bangla")

    fields = extract_fields(merged)
    timings["total"] = time.perf_counter() - t_start
    return {"image": Path(image_path).name, "fields": fields,
            "lines": merged, "timings": timings}


out = hybrid_ocr(sample, verbose=True)
print(json.dumps({k: v for k, v in out["fields"].items() if k != "_flags"},
                 ensure_ascii=False, indent=2))

if out["fields"]["_flags"]:
    print("\nflags (things that need a human look):")
    for fl in out["fields"]["_flags"]:
        print("  -", fl)

print("\ntimings (seconds):")
for k, v in out["timings"].items():
    print(f"  {k:<26} {v:.2f}" if isinstance(v, float) else f"  {k:<26} {v}")

---

## 8. Step 6 — benchmark on the 20-card set

Same metrics as `benchmark.ipynb`.

* **Field exact match** — the value matches ground truth character for character after whitespace normalisation. This is the number your supervisor cares about, especially for `nid` and `dob`.
* **CER (character error rate)** — edit distance divided by ground-truth length. Useful for names, where "almost right" still means something.
* **Latency p50 / p95** — median and the slow tail. p95 matters for a sign-up flow: it is roughly the worst experience a user will actually get.

First look at the ground-truth columns and map them to the field names used here.

In [ ]:
gt = pd.read_csv(GT_CSV)
print("columns in data.csv:", list(gt.columns))
print(f"{len(gt)} rows\n")
gt.head()

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))
from bench.common import (read_truth, exact_match, char_error_rate,
                          FIELDS, FIELD_KEYS, parse_nid, parse_dob, clean_text)

truth = read_truth()                  # {image_id: {field_key: value}}
print(f"{len(truth)} ground-truth rows")
print("fields:", FIELD_KEYS)
print("sample :", truth["110"])

In [ ]:
bench_cards = list_cards(N_BENCH_CARDS)

records, per_card, failures = [], [], []
for k, path in enumerate(bench_cards, 1):
    print(f"[{k}/{len(bench_cards)}] {path.name} ...", end=" ", flush=True)

    # One bad card must not throw away the whole run - at ~90s per card this
    # loop is half an hour of work. Record the failure and carry on; a
    # benchmark that reports "19/20, 1 crashed" is a more honest result.
    try:
        res = hybrid_ocr(path)
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {e}")
        failures.append({"image_id": path.stem, "error": f"{type(e).__name__}: {e}"})
        continue

    tm = res["timings"]
    print(f"{tm['total']:.1f}s ({tm['n_vl_crops']} VL crops)")

    row = {"image_id": path.stem, **tm}
    t = truth.get(path.stem)
    if t is None:
        print("   (no ground-truth row)")
    else:
        for f in FIELD_KEYS:
            p = res["fields"].get(f) or ""
            row[f"{f}_pred"]  = p
            row[f"{f}_true"]  = t[f]
            row[f"{f}_exact"] = int(exact_match(f, p, t[f]))
            row[f"{f}_cer"]   = char_error_rate(f, p, t[f])
    records.append(row)
    per_card.append(res)

    # checkpoint after every card, so an interrupt still leaves usable data
    pd.DataFrame(records).to_csv(OUT_DIR / "hybrid_per_card.csv", index=False)

bench = pd.DataFrame(records)
bench.to_csv(OUT_DIR / "hybrid_per_card.csv", index=False)
# Guard against llama.cpp prompt-cache contamination - see item 8.
# An absolute threshold does not survive a model swap (a crop costs ~10s on
# BF16 but ~2s on Q4_K_M), so compare each card against the run's own median
# instead: a card that is much faster than its peers was served from cache.
if len(records) >= 5:
    per_crop = bench.loc[bench["n_vl_crops"] > 0, "vl_bangla_crops"] /                bench.loc[bench["n_vl_crops"] > 0, "n_vl_crops"]
    med = per_crop.median()
    warm = bench.loc[per_crop.index[per_crop < 0.5 * med], "image_id"]
    print(f"median VL time per crop: {med:.2f}s")
    if len(warm):
        print("WARNING: served from a warm prompt cache, timing not valid for:",
              ", ".join(warm.astype(str)))
        print("         restart llama-server before the run to get clean numbers")

print()
print(f"{len(records)}/{len(bench_cards)} cards completed")
print(f"saved {OUT_DIR / 'hybrid_per_card.csv'}")

if failures:
    print()
    print("cards that failed (report these, do not silently drop them):")
    for f in failures:
        print(f"  - {f['image_id']}: {f['error']}")


In [ ]:
# ---- accuracy per field -------------------------------------------------
acc = pd.DataFrame([{
    "field": f,
    "exact_match_%": 100 * bench[f"{f}_exact"].mean(),
    "mean_CER": bench[f"{f}_cer"].mean(),
    "engine": "VL (Bangla)" if f.endswith("_bn") else "PP-OCR (English)",
} for f in FIELD_KEYS if f"{f}_exact" in bench.columns])

print("FIELD ACCURACY - hybrid pipeline\n")
print(acc.to_string(index=False, float_format=lambda v: f"{v:6.2f}"))

# ---- latency ------------------------------------------------------------
print("\n\nLATENCY (seconds per card, CPU only)\n")
lat = pd.DataFrame([{
    "stage": s,
    "p50": bench[s].quantile(0.50),
    "p95": bench[s].quantile(0.95),
    "mean": bench[s].mean(),
} for s in ["ppocr_detect_recognise", "vl_bangla_crops", "total"]])
print(lat.to_string(index=False, float_format=lambda v: f"{v:7.2f}"))

print(f"\nVL crops per card: median {bench['n_vl_crops'].median():.0f}, "
      f"max {bench['n_vl_crops'].max():.0f}")
share = bench["vl_bangla_crops"].sum() / bench["total"].sum() * 100
print(f"VL accounts for {share:.0f}% of total pipeline time")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

acc.plot.barh(x="field", y="exact_match_%", ax=ax[0], legend=False, color="#2a7ab0")
ax[0].set_xlim(0, 100); ax[0].set_xlabel("exact match %"); ax[0].set_ylabel("")
ax[0].set_title("Field accuracy")

ax[1].bar(["PP-OCR", "VL"],
          [bench["ppocr_detect_recognise"].mean(), bench["vl_bangla_crops"].mean()],
          color=["#2a7ab0", "#d1495b"])
ax[1].set_ylabel("seconds per card (mean)")
ax[1].set_title("Where the time goes")

plt.tight_layout(); plt.show()

---

## 9. Step 7 — the control: VL alone, doing everything

The hybrid is more moving parts than a single engine. It has to earn that. So run VL on its own over the whole card with the `Spotting:` prompt — find and read all text, English and Bangla — and compare.

Three possible outcomes, and each points somewhere different:

* **VL-only is as accurate and not much slower** → drop the hybrid, recommend VL alone. Simpler is better.
* **VL-only makes digit mistakes on `nid` or `dob`** → this is the expected result, and it is the strongest argument for the hybrid. Say so in the report with the exact examples.
* **VL-only is far slower** → also an argument for the hybrid, since VL then only handles a few small crops.

This run is slow. It defaults to 5 cards.

In [ ]:
def vl_only(image_path):
    img = Image.open(image_path).convert("RGB")
    t0 = time.perf_counter()
    # repeat_penalty is not optional here: at temp 0 with none, Spotting falls
    # into a token loop, burns all 1024 tokens and returns one usable line.
    raw = vl_ocr(img, task="spotting", max_tokens=1024)
    elapsed = time.perf_counter() - t0

    merged = [{"text": ln.strip(), "poly": np.array([[0, i * 20], [200, i * 20],
                                                     [200, i * 20 + 18], [0, i * 20 + 18]],
                                                    dtype=np.float32), "source": "vl"}
              for i, ln in enumerate(raw.splitlines()) if ln.strip()]
    return {"fields": extract_fields(merged), "raw": raw, "seconds": elapsed}


ctrl_rows = []
for k, path in enumerate(bench_cards[:N_CONTROL_CARDS], 1):
    print(f"[{k}/{N_CONTROL_CARDS}] {path.name} ...", end=" ", flush=True)
    r = vl_only(path)
    print(f"{r['seconds']:.1f}s")

    row = {"image_id": path.stem, "seconds": r["seconds"]}
    t = truth.get(path.stem)          # do NOT name this `truth` - it shadows the global
    if t is None:
        print("   (no ground-truth row)")
    else:
        for f in FIELD_KEYS:
            p = r["fields"].get(f) or ""
            row[f"{f}_pred"]  = p
            row[f"{f}_true"]  = t[f]
            row[f"{f}_exact"] = int(exact_match(f, p, t[f]))
            row[f"{f}_cer"]   = char_error_rate(f, p, t[f])
    ctrl_rows.append(row)

ctrl = pd.DataFrame(ctrl_rows)
ctrl.to_csv(OUT_DIR / "vl_only_per_card.csv", index=False)
ctrl[["image_id", "seconds"]]

In [ ]:
sub = bench[bench["image_id"].isin(ctrl["image_id"])]

compare = pd.DataFrame([{
    "field": f,
    "hybrid_exact_%":  100 * sub[f"{f}_exact"].mean(),
    "vl_only_exact_%": 100 * ctrl[f"{f}_exact"].mean() if f"{f}_exact" in ctrl else np.nan,
    "hybrid_CER":      sub[f"{f}_cer"].mean(),
    "vl_only_CER":     ctrl[f"{f}_cer"].mean() if f"{f}_cer" in ctrl else np.nan,
} for f in FIELD_KEYS if f"{f}_exact" in sub.columns])

print(f"HYBRID vs VL-ONLY  (same {len(ctrl)} cards)\n")
print(compare.to_string(index=False, float_format=lambda v: f"{v:7.2f}"))

print(f"\nseconds per card:  hybrid {sub['total'].mean():.1f}   "
      f"vl-only {ctrl['seconds'].mean():.1f}")

print("\nDigit fields are the ones to stare at - nid and dob.")
print("A generative model inventing one plausible digit is a KYC failure, ")
print("and it will not show up as low confidence anywhere.")

---

## 10. Errors worth looking at by hand

Numbers tell you *how much* is wrong. This cell tells you *what* is wrong, which is what decides the next experiment.

In [ ]:
print("WORST FIELDS, hybrid pipeline\n")
for f in FIELD_KEYS:
    if f"{f}_exact" not in bench.columns:
        continue
    bad = bench[bench[f"{f}_exact"] == 0]
    if bad.empty:
        continue
    print(f"--- {f}  ({len(bad)} of {len(bench)} wrong) ---")
    for _, r in bad.head(5).iterrows():
        print(f"  {r['image_id']}")
        print(f"     want: {r[f'{f}_true']!r}")
        print(f"     got : {r[f'{f}_pred']!r}   (CER {r[f'{f}_cer']:.2f})")
    print()

In [ ]:
# routing audit: Bangla characters that ended up in a PP-OCR line,
# or a line sent to VL that was plainly English all along
print("ROUTING AUDIT\n")
leaks = wasted = 0
for res in per_card:
    for ln in res["lines"]:
        if ln["source"] == "ppocr" and BANGLA_RANGE.search(ln.get("ppocr_text") or ""):
            leaks += 1
        if ln["source"] == "vl" and not BANGLA_RANGE.search(ln["text"] or "") and ln["text"]:
            wasted += 1

total_lines = sum(len(r["lines"]) for r in per_card)
print(f"lines total                    : {total_lines}")
print(f"possible Bangla kept as English: {leaks}")
print(f"English lines sent to VL       : {wasted}  (wasted time, and VL may paraphrase)")
print("\nIf either number is high, retune SCORE_THRESHOLD in section 4b.")

In [ ]:
import random
import html
from IPython.display import HTML, display

# ----------------------------------------------------------------------
# Show N random cards: the card on the left, the 6 extracted fields on
# the right. Boxes on the card: green = read by PaddleOCR, red = read by VL.
# ----------------------------------------------------------------------
N_SHOW = 5
SEED   = None       # set a number (e.g. 42) to get the same 5 cards every run

FIELD_ROWS = [
    ("name_en",   "Name (English)"),
    ("name_bn",   "Name (Bangla)"),
    ("father_bn", "Father"),
    ("mother_bn", "Mother"),
    ("dob",       "Date of birth"),
    ("nid",       "NID number"),
]


def card_to_data_url(path, lines, width=520):
    """Card image with line boxes drawn on it, as an embeddable picture."""
    img = cv2.imread(str(path))
    thick = max(1, img.shape[1] // 400)
    for ln in lines:
        colour = (0, 160, 0) if ln["source"] == "ppocr" else (0, 0, 220)   # BGR
        cv2.polylines(img, [np.asarray(ln["poly"]).astype(np.int32)], True, colour, thick)
    scale = width / img.shape[1]
    img = cv2.resize(img, (width, int(img.shape[0] * scale)), interpolation=cv2.INTER_AREA)
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode()


# ground truth is optional - only used if section 8 has been run
have_truth = "truth" in globals() and "exact_match" in globals()

# picked = random.Random(SEED).sample(cards, min(N_SHOW, len(cards)))
# WANTED = ["preproc_1", "preproc_2", "preproc_3", "preproc_4", "preproc_5"]
WANTED = ["data_1", "data_2", "data_3", "data_4", "data_5"]

by_stem = {p.stem: p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in IMG_EXTS}
missing = [name for name in WANTED if name not in by_stem]
if missing:
    print("NOT FOUND in", IMAGE_DIR, ":", missing)
picked = [by_stem[name] for name in WANTED if name in by_stem]
print("cards picked:", [p.name for p in picked], "\n")

shown_results = []      # kept so the next cell can show the timings
html_blocks   = []      # kept so the whole showcase can be saved as a page

for path in picked:
    try:
        res = hybrid_ocr(path)
    except Exception as e:
        print(f"{path.name}: FAILED - {type(e).__name__}: {e}")
        continue

    shown_results.append({"card": path.name, **res["timings"]})

    fields = res["fields"]
    how    = fields.get("_how", {})
    t      = truth.get(path.stem) if have_truth else None

    rows_html = ""
    for key, label in FIELD_ROWS:
        value = fields.get(key)
        shown = html.escape(str(value)) if value else "<i style='opacity:.5'>not found</i>"
        check = ""
        if t is not None:
            if exact_match(key, value or "", t[key]):
                check = "<span style='color:#2e9e44'>✓</span>"
            else:
                check = (f"<span style='color:#d64545'>✗</span> "
                         f"<span style='opacity:.65'>expected: {html.escape(str(t[key]))}</span>")
        rows_html += (
            "<tr>"
            f"<td style='padding:6px 12px;opacity:.7;white-space:nowrap'>{label}</td>"
            f"<td style='padding:6px 12px;font-size:15px'><b>{shown}</b></td>"
            f"<td style='padding:6px 12px;opacity:.55;font-size:12px'>{html.escape(how.get(key, '-'))}</td>"
            f"<td style='padding:6px 12px;font-size:13px'>{check}</td>"
            "</tr>"
        )

    tm = res["timings"]
    block = f"""
    <div style="display:flex;gap:28px;align-items:flex-start;margin:22px 0;
                padding-bottom:22px;border-bottom:1px solid rgba(128,128,128,.3);
                font-family:'Nirmala UI','Vrinda','Noto Sans Bengali',sans-serif;">
      <div style="flex:0 0 auto">
        <img src="{card_to_data_url(path, res['lines'])}"
             style="max-width:520px;border:1px solid rgba(128,128,128,.4);border-radius:4px">
        <div style="font-size:12px;opacity:.65;margin-top:6px">
          <b>{html.escape(path.name)}</b> &nbsp;·&nbsp;
          <span style="color:#2e9e44">green</span> = PaddleOCR &nbsp;·&nbsp;
          <span style="color:#d64545">red</span> = VL &nbsp;·&nbsp;
          {tm['total']:.1f}s
        </div>
      </div>
      <table style="border-collapse:collapse;flex:1 1 auto">
        <tr style="border-bottom:1px solid rgba(128,128,128,.4)">
          <th style="text-align:left;padding:6px 12px">Field</th>
          <th style="text-align:left;padding:6px 12px">Extracted</th>
          <th style="text-align:left;padding:6px 12px">How found</th>
          <th style="text-align:left;padding:6px 12px">{'Check' if t is not None else ''}</th>
        </tr>
        {rows_html}
      </table>
    </div>
    """
    html_blocks.append(block)
    display(HTML(block))

# ----------------------------------------------------------------------
# Save to disk so the results survive a kernel restart
# ----------------------------------------------------------------------
SHOWCASE_CSV  = OUT_DIR / "showcase_timings.csv"
SHOWCASE_HTML = OUT_DIR / "showcase.html"

pd.DataFrame(shown_results).to_csv(SHOWCASE_CSV, index=False)

page = f"""<!doctype html>
<html><head><meta charset="utf-8"><title>NID OCR showcase</title></head>
<body style="font-family:'Nirmala UI','Vrinda','Noto Sans Bengali',sans-serif;
             max-width:1200px;margin:24px auto;padding:0 16px">
<h2>Hybrid NID OCR - {len(html_blocks)} random cards</h2>
<p style="opacity:.65">Saved {time.strftime('%Y-%m-%d %H:%M')} ·
green boxes = PaddleOCR · red boxes = VL</p>
{''.join(html_blocks)}
</body></html>"""
SHOWCASE_HTML.write_text(page, encoding="utf-8")

print(f"saved {SHOWCASE_CSV}")
print(f"saved {SHOWCASE_HTML}  (open it in any browser)")

In [ ]:
# ----------------------------------------------------------------------
# Time taken per card in the showcase cell above.
# Uses this session's results if they exist, otherwise the saved CSV -
# so this cell still works after a kernel restart without re-running.
# ----------------------------------------------------------------------
SHOWCASE_CSV = OUT_DIR / "showcase_timings.csv"
TIMING_PNG   = OUT_DIR / "showcase_timings.png"

if "shown_results" in globals() and shown_results:
    tdf, source = pd.DataFrame(shown_results), "this session"
elif SHOWCASE_CSV.exists():
    tdf, source = pd.read_csv(SHOWCASE_CSV), f"saved file {SHOWCASE_CSV.name}"
else:
    tdf, source = None, None
    print("No timings yet - run the showcase cell above first.")

if tdf is not None:
    tdf["other"] = (tdf["total"] - tdf["ppocr_detect_recognise"]
                    - tdf["vl_bangla_crops"]).clip(lower=0)
    tdf["vl_per_crop"] = tdf["vl_bangla_crops"] / tdf["n_vl_crops"].replace(0, np.nan)

    table = tdf[["card", "total", "ppocr_detect_recognise", "vl_bangla_crops",
                 "other", "n_lines", "n_vl_crops", "vl_per_crop"]].rename(columns={
        "card": "Card",
        "total": "Total (s)",
        "ppocr_detect_recognise": "PaddleOCR (s)",
        "vl_bangla_crops": "VL (s)",
        "other": "Other (s)",
        "n_lines": "Lines",
        "n_vl_crops": "VL crops",
        "vl_per_crop": "VL per crop (s)",
    })
    print(f"TIME PER CARD   (source: {source})\n")
    print(table.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

    print(f"\naverage per card : {tdf['total'].mean():.2f} s")
    print(f"fastest          : {tdf['total'].min():.2f} s  ({tdf.loc[tdf['total'].idxmin(), 'card']})")
    print(f"slowest          : {tdf['total'].max():.2f} s  ({tdf.loc[tdf['total'].idxmax(), 'card']})")
    share = tdf["vl_bangla_crops"].sum() / tdf["total"].sum() * 100
    print(f"VL share of time : {share:.0f}%")

    # stacked bar: where the time goes on each card
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(tdf) + 1.6))
    y = np.arange(len(tdf))
    ax.barh(y, tdf["ppocr_detect_recognise"], color="#2a7ab0", label="PaddleOCR")
    ax.barh(y, tdf["vl_bangla_crops"], left=tdf["ppocr_detect_recognise"],
            color="#d1495b", label="VL")
    ax.barh(y, tdf["other"],
            left=tdf["ppocr_detect_recognise"] + tdf["vl_bangla_crops"],
            color="#b0b0b0", label="Other")
    for yi, total in zip(y, tdf["total"]):
        ax.text(total + 0.2, yi, f"{total:.1f}s", va="center", fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(tdf["card"])
    ax.invert_yaxis()
    ax.set_xlabel("seconds")
    ax.set_xlim(0, tdf["total"].max() * 1.15)
    ax.set_title("Time per card (CPU only)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(TIMING_PNG, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved {TIMING_PNG}")